## Importing Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import clickhouse_connect

### Define Parameters

In [2]:
MONTH = pd.Period("2026-07")  # <-- only thing to change each run

START, END = MONTH.start_time.date(), (MONTH + 1).start_time.date()

def date_filter(col: str) -> str:
    """SQL WHERE clause for the analysis month, on the given timestamp column."""
    return f"WHERE {col} >= '{START}' AND {col} < '{END}'"

# KYC dump is for the same month as the IMEI data
KYC_TAG = MONTH.strftime("%m_%y")          # e.g. "07_26"
MONTH_TAG = MONTH.strftime("%Y-%m")        # e.g. "2026-07"

KYC_DIR = Path("/Volumes/E$/KYC/Merged Clean Dumps/2026")
OUT_DIR = Path("/Volumes/E$/CEIR/Clean Dumps")
MCC_CSV = Path("/Users/wmuheki/Documents/Projects/Analytics/ceir/clean_dumps/MCC_Each_country.csv")

print(date_filter("imei_first_seen"))
print(f"KYC files: df_{KYC_TAG}.parquet / df_NID_{KYC_TAG}.parquet")

WHERE imei_first_seen >= '2026-07-01' AND imei_first_seen < '2026-08-01'
KYC files: df_07_26.parquet / df_NID_07_26.parquet


### Define Clickhouse Connect - Single Reused

In [3]:
client = clickhouse_connect.get_client(
    host='192.168.1.95',
    port=8123,
    username='default',
    password='',
    settings={
        'max_memory_usage': 4000000000,  # 4GB max
        'max_threads': 2,
        'priority': 5
    }
)

### Helper Functions

In [4]:
def attach_kyc(target_df: pd.DataFrame, nid_df: pd.DataFrame, fallback_df: pd.DataFrame) -> pd.DataFrame:
    """Attach KYC on msisdn: NID-registered values win, fallback fills the gaps."""
    out = target_df.merge(nid_df, on='msisdn', how='left')
    out = out.merge(fallback_df, on='msisdn', how='left', suffixes=('', '_fb'))

    overlap = ['id_type', 'id_number', 'prefix', 'mno']  # columns present in both KYC frames
    for col in overlap:
        out[col] = out[col].astype('object').fillna(out[f'{col}_fb'].astype('object'))

    out = out.drop(columns=[f'{col}_fb' for col in overlap])

    for col in ['id_type', 'prefix', 'mno']:
        out[col] = out[col].astype('category')
    return out


IMSI_PREFIX_MNO = {
    '64110': 'MTN',
    '64120': 'HAMILTON',
    '64108': 'TALKIO',
    '64101': 'AIRTEL',
    '64122': 'AIRTEL',
}

def fill_mno_from_imsi(df: pd.DataFrame) -> pd.DataFrame:
    """Where mno is missing, derive it from the IMSI prefix."""
    mask = df['mno'].isna()
    imsi = df['imsi'].astype('string')

    if isinstance(df['mno'].dtype, pd.CategoricalDtype):
        new_cats = [m for m in set(IMSI_PREFIX_MNO.values()) if m not in df['mno'].cat.categories]
        if new_cats:
            df['mno'] = df['mno'].cat.add_categories(new_cats)

    for prefix, operator in IMSI_PREFIX_MNO.items():
        df.loc[mask & imsi.str.startswith(prefix), 'mno'] = operator
    return df


def add_country(df: pd.DataFrame, mcc_lookup: pd.DataFrame) -> pd.DataFrame:
    """Extract MCC (first 3 digits of IMSI) and merge in the country name."""
    df['imsi'] = df['imsi'].astype('string')
    df['mcc'] = (
        df['imsi']
        .str.replace(r'\D+', '', regex=True)  # keep digits only
        .str.slice(0, 3)
    )
    df = df.merge(mcc_lookup[['mcc', 'country']], how='left', on='mcc')
    df['country'] = df['country'].fillna('UNKNOWN')
    return df


def save_monthly_parquet(df: pd.DataFrame, category: str) -> Path:
    """Save df to <OUT_DIR>/<category>/<category.lower()>_<YYYY-MM>.parquet."""
    out_path = OUT_DIR / category / f"{category.lower()}_{MONTH_TAG}.parquet"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_path)
    print(f"Saved {len(df):,} rows -> {out_path}")
    return out_path

### Importing KYC data

In [5]:
df = pd.read_parquet(KYC_DIR / f"df_{KYC_TAG}.parquet")
df_NID = pd.read_parquet(KYC_DIR / f"df_NID_{KYC_TAG}.parquet")

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/E$/KYC/Merged Clean Dumps/2026/df_07_26.parquet'

In [ ]:
df.head()

In [ ]:
df_NID.head()

In [ ]:
# Keep only local-format MSISDNs (10 digits, leading 0), then convert to 256 format.
# NOTE: this deliberately drops rows already in international 256... format (12 digits);
# widen the filter to .isin([10, 12]) if those should be kept.
df = df[df['msisdn'].astype(str).str.len() == 10].copy()
df_NID = df_NID[df_NID['msisdn'].astype(str).str.len() == 10].copy()

df['msisdn'] = df['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)
df_NID['msisdn'] = df_NID['msisdn'].astype(str).str.replace(r'^0', '256', regex=True)

# Drop unnecessary columns to save memory
df = df.drop(columns=['surname', 'first_name'])
df_NID = df_NID.drop(columns=['surname', 'first_name'])

### Importing MCC Lookup (loaded once, shared by all datasets)

In [ ]:
mcc_lu = pd.read_csv(MCC_CSV, dtype=str)

mcc_lu = mcc_lu.rename(columns={"MCC": "mcc", "Country": "country"})
mcc_lu["mcc"] = mcc_lu["mcc"].astype("string").str.strip()
mcc_lu["country"] = mcc_lu["country"].astype("string").str.strip()
mcc_lu = mcc_lu.drop_duplicates(subset=["mcc"])

mcc_lu.head()

### Importing GSMA data

In [ ]:
gsma_query = """
SELECT
    tac,
    oem,
    brand,
    model,
    marketing_name,
    device_type,
    os_family,
    os_version,
    sim_slots,
    has_2g,
    has_3g,
    has_4g,
    has_5g,
    year_released
FROM ceir.gsma_devices
"""
gsma_df = client.query_df(gsma_query)
len(gsma_df)

In [ ]:
gsma_df.head()

### Importing Fake IMEIs Table

Fake IMEIs are intentionally **not** merged with GSMA: their TACs are typically
unallocated, so the merge would produce mostly nulls/noise.

In [ ]:
fake_query = f"SELECT * FROM ceir_gold.imeis_fake_v {date_filter('imei_first_seen')}"
fake_df = client.query_df(fake_query)
len(fake_df)

In [ ]:
fake_df.head()

In [ ]:
# Drop empty device_type column and re-arrange columns
fake_df = fake_df.drop(columns=['device_type'])
fake_df = fake_df[['imei_first_seen', 'last_seen', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

In [ ]:
# Enrich: KYC coalesce -> MNO from IMSI where missing -> country from MCC
fake_df = attach_kyc(fake_df, df_NID, df)
fake_df = fill_mno_from_imsi(fake_df)
fake_df = add_country(fake_df, mcc_lu)

In [ ]:
fake_df.head()

### Importing Genuine IMEIs Table

In [ ]:
genuine_query = f"SELECT * FROM ceir_gold.imeis_genuine_v {date_filter('imei_first_seen')}"
genuine_df = client.query_df(genuine_query)
len(genuine_df)

In [ ]:
# Drop device_type — regenerated from GSMA TAC data below
genuine_df = genuine_df.drop(columns=['device_type'])

# Derive TAC (first 8 digits of the IMEI) and re-arrange columns
genuine_df['tac'] = genuine_df['imei'].astype(str).str[:8]
genuine_df = genuine_df[['imei_first_seen', 'last_seen', 'tac', 'imei', 'imei_status', 'imsi', 'msisdn', 'rat', 'cgi']]

# Merge with GSMA data to get device details
genuine_df = genuine_df.merge(gsma_df, on='tac', how='left')

In [ ]:
# Enrich: KYC coalesce -> MNO from IMSI where missing -> country from MCC
genuine_df = attach_kyc(genuine_df, df_NID, df)
genuine_df = fill_mno_from_imsi(genuine_df)
genuine_df = add_country(genuine_df, mcc_lu)

In [ ]:
# Convert GSMA numeric columns to nullable integers (removes decimal points)
int_cols = ['sim_slots', 'has_2g', 'has_3g', 'has_4g', 'has_5g', 'year_released']
genuine_df[int_cols] = genuine_df[int_cols].astype('Int64')

In [ ]:
genuine_df.head()

### Cloned IMEIs

In [ ]:
# Filter in SQL so only the analysis month is pulled (the view's timestamp
# column is first_detected_at, renamed to imei_first_seen after loading)
clone_query = f"SELECT * FROM ceir_gold.cloned_imeis_v {date_filter('first_detected_at')}"
clone_df = client.query_df(clone_query)
len(clone_df)

In [ ]:
# Drop device_type (regenerated from GSMA) and the array columns
clone_df = clone_df.drop(columns=['device_type', 'imsis', 'msisdns'])

# Rename for consistency with the other datasets
clone_df = clone_df.rename(columns={'first_detected_at': 'imei_first_seen', 'last_change_at': 'last_seen'})

# Derive TAC and re-arrange columns
clone_df['tac'] = clone_df['imei'].astype(str).str[:8]
clone_df = clone_df[['imei_first_seen', 'last_seen', 'tac', 'imei', 'imsi_count', 'msisdn_count']]

# Merge with GSMA data to get device details
clone_df = clone_df.merge(gsma_df, on='tac', how='left')

In [ ]:
clone_df.head()

### Export — filenames derived from MONTH, nothing to edit

In [ ]:
save_monthly_parquet(fake_df, "Fake")
save_monthly_parquet(genuine_df, "Genuine")
save_monthly_parquet(clone_df, "Cloned")

### Cleanup

In [ ]:
client.close()